<a href="https://colab.research.google.com/github/Pranayshukla0610/ML-projects-portfolio/blob/main/LinearRegression_On_Taxi_Duration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import zipfile

In [ ]:
!file /content/nyc-taxi-trip-duration.zip

/content/nyc-taxi-trip-duration.zip: Zip archive data, at least v4.5 to extract, compression method=deflate


In [ ]:
!pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle competitions download -c nyc-taxi-trip-duration

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [ ]:
!unzip nyc-taxi-trip-duration.zip

Archive:  nyc-taxi-trip-duration.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of nyc-taxi-trip-duration.zip or
        nyc-taxi-trip-duration.zip.zip, and cannot find nyc-taxi-trip-duration.zip.ZIP, period.


In [ ]:
df = pd.read_csv('/content/train.zip')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.zip'

In [ ]:
df = df.dropna()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

In [ ]:
from geopy.distance import geodesic

def calculate_distance(row):
  return geodesic(
      (row['pickup_latitude'],row['pickup_longitude']),
      (row['dropoff_latitude'],row['dropoff_longitude'])
  ).km

df['distance'] = df.apply(calculate_distance, axis=1)

In [ ]:
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.dayofweek

In [ ]:
df['is_rush_hour'] = df['hour'].apply(lambda x : 1 if 7 <= x <= 10 or 16 <= x <= 19 else 0)

In [ ]:
df.head()

In [ ]:
sns.scatterplot(x='distance', y='trip_duration',data=df)
plt.show()

In [ ]:
sns.boxplot(x= 'hour', y='trip_duration', data = df)

In [ ]:
numeric_df = df.select_dtypes(include=['number'])
sns.heatmap(numeric_df.corr(), annot=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df[['distance','hour','is_rush_hour']]
y = df['trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = LinearRegression()
model.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test)

print('MAE:',mean_squared_error(y_test,y_pred))
print('R2:',r2_score(y_test,y_pred))

In [ ]:
plt.scatter(y_test, y_pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st

st.title("Ride Duration Predictor")

distance = st.slider("Distance (Km)",1,50)
hour = st.slider("Hour",0,23)
rush = st.selectbox("Rush_Hour",[0,1])

prediction = model.predict([[distance, hour, rush]])

st.write(f"Predicted Ride Duration: {prediction[0]}")
